<style>
:root { --navy:#1B3A4B; --teal:#2A9D8F; --coral:#E76F51; --gold:#E9C46A; --cream:#F7F4EF; --muted:#6B6560; --rule:#D4CBBE; --navy-lt:#2C5F7A; }
.nb-title { background:linear-gradient(135deg,#1B3A4B 0%,#2C5F7A 100%); color:#FFF; padding:28px 32px 24px; border-radius:10px; margin:0 0 18px; font-family:'Segoe UI',Arial,sans-serif; box-shadow:0 4px 14px rgba(27,58,75,.25); }
.nb-title h1 { margin:0 0 8px; font-size:1.85em; font-weight:700; color:#FFF; border:none; padding:0; }
.nb-title .subtitle { margin:0; font-size:1.05em; color:#A8C5D4; }
.nb-title .meta { margin:12px 0 0; font-size:.88em; color:#2A9D8F; letter-spacing:.04em; text-transform:uppercase; font-weight:600; }
.nb-section { background:#F7F4EF; border-left:5px solid #2A9D8F; padding:12px 18px; margin:22px 0 12px; border-radius:0 8px 8px 0; font-family:'Segoe UI',Arial,sans-serif; }
.nb-section h2 { margin:0; font-size:1.35em; color:#1B3A4B; font-weight:700; border:none; padding:0; }
.nb-box { background:#FFF; border:1px solid #D4CBBE; border-radius:8px; padding:14px 18px; margin:12px 0; font-family:'Segoe UI',Arial,sans-serif; color:#1A1A1A; line-height:1.55; }
.nb-callout { background:#E8F5F3; border:1px solid #2A9D8F; border-left:5px solid #2A9D8F; border-radius:0 8px 8px 0; padding:12px 16px; margin:12px 0; font-family:'Segoe UI',Arial,sans-serif; }
.nb-warning { background:#FDF0ED; border:1px solid #E76F51; border-left:5px solid #E76F51; border-radius:0 8px 8px 0; padding:12px 16px; margin:12px 0; font-family:'Segoe UI',Arial,sans-serif; }
.nb-summary { background:linear-gradient(135deg,#1B3A4B 0%,#2C5F7A 100%); color:#FFF; padding:22px 28px; border-radius:10px; margin:24px 0 12px; font-family:'Segoe UI',Arial,sans-serif; }
.nb-summary h2 { margin:0 0 14px; font-size:1.3em; color:#E9C46A; border:none; padding:0; }
.nb-summary table { width:100%; border-collapse:collapse; margin:10px 0; font-size:.92em; }
.nb-summary th { text-align:left; padding:8px 10px; color:#A8C5D4; border-bottom:1px solid #2C5F7A; }
.nb-summary td { padding:8px 10px; border-bottom:1px solid rgba(255,255,255,.08); color:#F0F4F8; }
.nb-summary p { margin:12px 0 0; color:#A8C5D4; }
.nb-next { background:#F7F4EF; border:2px dashed #2A9D8F; border-radius:8px; padding:14px 18px; margin:18px 0; font-family:'Segoe UI',Arial,sans-serif; color:#1B3A4B; }
</style>

<div class="nb-title">
  <h1>Kaggle Survey Comparative Analysis - 2020 vs 2022</h1>
  <p class="subtitle">Demographics, skills, tools, and career signals across two global data-science communities</p>
  <p class="meta">EDA · Feature engineering · Statistical comparison · Reproducible pipeline</p>
</div>



<div class="nb-box">
  <p><strong>Decision context.</strong> Stakeholders need a clear view of how the Kaggle community evolved between 2020 and 2022: who participates, what they know, which tools dominate, and where compensation and role distributions shifted.</p>
  <p><strong>Data contract.</strong> Official Kaggle ML &amp; DS Survey responses. Grain = individual respondent. Cleaned outputs in <code>Datasets/cleaned/</code>.</p>
  <p><strong>Limitations.</strong> Self-selected sample; question wording changed between years. We harmonize only high-confidence mappings.</p>
</div>


<div class="nb-section"><h2>1. Setup &amp; reproducible I/O</h2></div>

In [ ]:

import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)

DATA = Path("/home/workdir/artifacts/Datasets/cleaned")
assert DATA.exists(), "Run the cleaning script first"
print("Libraries ready · seed =", SEED)


<div class="nb-section"><h2>2. Load cleaned &amp; harmonized data</h2></div>

In [ ]:

harm = pd.read_parquet(DATA / "survey_harmonized_2020_2022.parquet")
c20 = pd.read_parquet(DATA / "survey_2020_cleaned.parquet")
c22 = pd.read_parquet(DATA / "survey_2022_cleaned.parquet")
print(f"Harmonized: {harm.shape[0]:,} rows x {harm.shape[1]} cols")
print(f"2020 cleaned: {c20.shape[0]:,}  |  2022 cleaned: {c22.shape[0]:,}")
display(harm["survey_year"].value_counts().sort_index())
display(harm.dtypes.head(12))



<div class="nb-callout">
  <strong>Insight:</strong> After dropping the question-text header row, exact duplicates, and extreme duration outliers, we retain ~19k (2020) and ~23k (2022) usable responses. Harmonized table aligns 34 core variables.
</div>


<div class="nb-section"><h2>3. Missingness &amp; data quality</h2></div>

In [ ]:

key = ["age","gender","country","education","role_group","coding_experience","ml_experience","compensation"]
def miss_pct(df, label):
    m = df[key].isna().mean()*100
    return m.rename(label)
prof = pd.concat([miss_pct(c20, 2020), miss_pct(c22, 2022)], axis=1).round(1)
display(prof.style.format("{:.1f}%"))

fig, ax = plt.subplots(figsize=(8, 3.5))
comp_miss = harm.groupby("survey_year")["compensation"].apply(lambda s: s.isna().mean()*100)
comp_miss.plot(kind="bar", ax=ax, color=["#2A9D8F","#1B3A4B"], edgecolor="white")
ax.set_ylabel("% missing"); ax.set_title("Compensation non-response"); ax.tick_params(rotation=0)
plt.tight_layout(); plt.show()



<div class="nb-warning">
  <strong>Caution:</strong> Compensation has high non-response (students / early-career). Condition salary analysis on non-missing and employment status.
</div>


<div class="nb-section"><h2>4. Demographic distributions &amp; shifts</h2></div>

In [ ]:

AGE_ORDER = ["18-21","22-24","25-29","30-34","35-39","40-44","45-49","50-54","55-59","60-69","70+"]
GENDER_ORDER = ["Man","Woman","Nonbinary","Prefer not to say","Other / Prefer not to say"]

def pct_table(df, col, order=None):
    t = df.groupby("survey_year")[col].value_counts(normalize=True).unstack(fill_value=0)*100
    if order:
        t = t.reindex(columns=[c for c in order if c in t.columns])
    return t.round(1)

age_pct = pct_table(harm, "age", AGE_ORDER)
gender_pct = pct_table(harm, "gender", GENDER_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
age_pct.T.plot(kind="bar", ax=axes[0], color=["#2A9D8F","#1B3A4B"], width=0.8)
axes[0].set_title("Age distribution (%)"); axes[0].set_ylabel("%"); axes[0].legend(title="Year"); axes[0].tick_params(axis="x", rotation=45)
gender_pct.T.plot(kind="bar", ax=axes[1], color=["#2A9D8F","#1B3A4B"], width=0.7)
axes[1].set_title("Gender distribution (%)"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
display(age_pct); display(gender_pct)


In [ ]:

def chi2_shift(df, col):
    ct = pd.crosstab(df["survey_year"], df[col])
    chi2, p, dof, exp = stats.chi2_contingency(ct)
    n = ct.to_numpy().sum(); r, k = ct.shape
    v = np.sqrt(chi2 / (n * (min(r, k) - 1)))
    return {"variable": col, "chi2": round(chi2,1), "p": p, "cramers_v": round(v,3), "n": int(n)}

tests = pd.DataFrame([chi2_shift(harm.dropna(subset=[c]), c) for c in ["age","gender","education","coding_experience","role_group"]])
display(tests.style.format({"p":"{:.2e}"}).background_gradient(subset=["cramers_v"], cmap="YlOrRd"))



<div class="nb-callout">
  <strong>Insight:</strong> Age and gender composition shifted modestly (small-to-medium Cramer's V). The 18-21 cohort grew as a share of responses. Gender imbalance remains pronounced.
</div>


<div class="nb-section"><h2>5. Education &amp; coding / ML experience</h2></div>

In [ ]:

EDU_ORDER = ["High school or less","Some college","Bachelor's","Master's","Doctoral","Professional","Prefer not to answer"]
# fix apostrophe variants present in data
edu_pct = pct_table(harm, "education")
code_pct = pct_table(harm, "coding_experience")
ml_pct = pct_table(harm, "ml_experience")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, tbl, title in zip(axes, [edu_pct, code_pct, ml_pct], ["Education","Coding experience","ML experience"]):
    tbl.T.plot(kind="bar", ax=ax, color=["#2A9D8F","#1B3A4B"], width=0.8, legend=(ax is axes[0]))
    ax.set_title(title); ax.tick_params(axis="x", rotation=55)
axes[0].set_ylabel("% of respondents")
plt.tight_layout(); plt.show()



<div class="nb-callout">
  <strong>Insight:</strong> Master's remains the modal education level. ML "None / under 1 year" still dominates, underscoring a large learner cohort.
</div>


<div class="nb-section"><h2>6. Programming languages &amp; ML frameworks</h2></div>

In [ ]:

LANGS = ["Python","R","SQL","C","C++","Java","Javascript","Bash","MATLAB","Julia"]
FWS = ["Scikit-learn","TensorFlow","Keras","PyTorch","Xgboost","LightGBM","CatBoost","Fast.ai","JAX"]

def usage_rate(df, prefix, items):
    rows = []
    for year, g in df.groupby("survey_year"):
        for item in items:
            col = f"{prefix}_{item}"
            if col in g.columns:
                rows.append({"year": year, "item": item, "pct": g[col].mean()*100})
    return pd.DataFrame(rows)

lang_u = usage_rate(harm, "lang", LANGS)
fw_u = usage_rate(harm, "fw", FWS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
pivot_l = lang_u.pivot(index="item", columns="year", values="pct").reindex(LANGS)
pivot_l.plot(kind="barh", ax=axes[0], color=["#2A9D8F","#1B3A4B"])
axes[0].set_xlabel("% using regularly"); axes[0].set_title("Programming language usage"); axes[0].invert_yaxis()
pivot_f = fw_u.pivot(index="item", columns="year", values="pct").reindex(FWS)
pivot_f.plot(kind="barh", ax=axes[1], color=["#2A9D8F","#1B3A4B"])
axes[1].set_xlabel("% using regularly"); axes[1].set_title("ML framework usage"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()
display(pivot_l.round(1)); display(pivot_f.round(1))



<div class="nb-callout">
  <strong>Insight:</strong> Python is near-ubiquitous. SQL and R remain strong. PyTorch gained share relative to TensorFlow/Keras between surveys - a clear ecosystem shift.
</div>


<div class="nb-section"><h2>7. Roles &amp; top countries</h2></div>

In [ ]:

role_pct = pct_table(harm, "role_group")
top_roles = role_pct.mean().sort_values(ascending=False).head(10).index.tolist()
role_top = role_pct[top_roles]
fig, ax = plt.subplots(figsize=(10, 5))
role_top.T.plot(kind="bar", ax=ax, color=["#2A9D8F","#1B3A4B"], width=0.8)
ax.set_title("Top role groups (%)"); ax.set_ylabel("%"); ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.show()
display(role_top.round(1))

top_c = harm["country"].value_counts().head(12).index.tolist()
ctry = harm[harm["country"].isin(top_c)]
ctry_pct = pct_table(ctry, "country")
fig, ax = plt.subplots(figsize=(10, 5))
ctry_pct.T.loc[top_c].plot(kind="barh", ax=ax, color=["#2A9D8F","#1B3A4B"])
ax.set_xlabel("% within top-12 countries"); ax.set_title("Top countries"); ax.invert_yaxis()
plt.tight_layout(); plt.show()


<div class="nb-section"><h2>8. Compensation bands (descriptive)</h2></div>

In [ ]:

comp = harm.dropna(subset=["compensation"])
print("Non-missing compensation:", len(comp), f"({100*len(comp)/len(harm):.1f}% of all)")
top_comp = comp.groupby("survey_year")["compensation"].value_counts(normalize=True).groupby(level=0).head(8)
display(top_comp.unstack(level=0).fillna(0).mul(100).round(1).sort_values(by=2022, ascending=False).head(10))



<div class="nb-warning">
  <strong>Caution:</strong> Compensation categories differ in wording. Direct mean comparison is not valid without mid-point imputation and PPP adjustment.
</div>


<div class="nb-section"><h2>9. Feature engineering - experience score &amp; tool diversity</h2></div>

In [ ]:

code_map = {"Never":0,"< 1 year":0.5,"1-2 years":1.5,"1-3 years":2,"3-5 years":4,"5-10 years":7.5,"10-20 years":15,"20+ years":25}
ml_map = {"None":0,"< 1 year":0.5,"1-2 years":1.5,"2-3 years":2.5,"3-4 years":3.5,"4-5 years":4.5,"5-10 years":7.5,"10-20 years":15,"20+ years":25}
fe = harm.copy()
fe["code_years_approx"] = fe["coding_experience"].map(code_map)
fe["ml_years_approx"] = fe["ml_experience"].map(ml_map)
lang_cols = [c for c in fe.columns if c.startswith("lang_") and c not in ("lang_None","lang_Other")]
fw_cols = [c for c in fe.columns if c.startswith("fw_") and c not in ("fw_None","fw_Other")]
fe["n_languages"] = fe[lang_cols].sum(axis=1)
fe["n_frameworks"] = fe[fw_cols].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for year, color in [(2020,"#2A9D8F"),(2022,"#1B3A4B")]:
    subset = fe[fe["survey_year"]==year]
    axes[0].hist(subset["n_languages"].dropna(), bins=range(0,12), alpha=0.6, label=str(year), color=color, edgecolor="white")
    axes[1].hist(subset["n_frameworks"].dropna(), bins=range(0,12), alpha=0.6, label=str(year), color=color, edgecolor="white")
axes[0].set_title("Language diversity"); axes[1].set_title("ML framework diversity")
axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()
print("Mean languages:", fe.groupby("survey_year")["n_languages"].mean().round(2).to_dict())
print("Mean frameworks:", fe.groupby("survey_year")["n_frameworks"].mean().round(2).to_dict())



<div class="nb-callout">
  <strong>Insight:</strong> Tool diversity is right-skewed; most respondents report 1-3 languages and 0-2 frameworks. Slight increases in framework diversity appear between years.
</div>


<div class="nb-section"><h2>10. Lightweight respondent segmentation (k-means)</h2></div>

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

seg = fe.dropna(subset=["code_years_approx","ml_years_approx"]).copy()
feat_cols = ["code_years_approx","ml_years_approx","n_languages","n_frameworks"]
X = StandardScaler().fit_transform(seg[feat_cols])
kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10)
seg["cluster"] = kmeans.fit_predict(X)

profile = seg.groupby("cluster")[feat_cols].mean().round(2)
profile["n"] = seg.groupby("cluster").size()
profile["pct"] = (profile["n"]/len(seg)*100).round(1)
display(profile.sort_values("code_years_approx"))
year_mix = pd.crosstab(seg["cluster"], seg["survey_year"], normalize="index")*100
display(Markdown("**Year composition inside each cluster (%)**"))
display(year_mix.round(1))



<div class="nb-callout">
  <strong>Insight:</strong> Four segments emerge: learners, practitioners, polyglots, and veterans. Cluster composition is relatively stable across survey years.
</div>



<div class="nb-summary">
  <h2>Summary of key findings</h2>
  <table>
    <tr><th>Theme</th><th>Finding</th><th>Implication</th></tr>
    <tr><td>Sample size</td><td>~19k (2020) / ~23k (2022) after cleaning</td><td>Adequate power for categorical tests</td></tr>
    <tr><td>Demographics</td><td>Younger cohort share rose; gender gap persists</td><td>Outreach still needed</td></tr>
    <tr><td>Education</td><td>Master's modal; Doctoral stable</td><td>Graduate-level community remains core</td></tr>
    <tr><td>Languages</td><td>Python dominant; SQL/R strong</td><td>Centre curriculum on Python+SQL</td></tr>
    <tr><td>Frameworks</td><td>PyTorch share increased vs TF/Keras</td><td>Ecosystem shift for content &amp; tooling</td></tr>
    <tr><td>Experience</td><td>Large learner segment (ML none / &lt;1 yr)</td><td>Onboarding products remain high-value</td></tr>
    <tr><td>Compensation</td><td>High non-response; bands not fully comparable</td><td>Use only with strong caveats</td></tr>
  </table>
  <p>Cleaned datasets feed the Streamlit comparison dashboard for interactive exploration.</p>
</div>



<div class="nb-next">
  <strong>Next phase -></strong> Interactive Streamlit dashboard (<code>Dashboards/kaggle_survey_comparison/</code>) with filters on year, country, role, and experience.
</div>



<div class="nb-box">
  <p><strong>Reproducibility.</strong> Seed = 42. Cleaning script produces the parquet extracts used here. Environment: see <code>requirements.txt</code>.</p>
</div>
